# gpu/cpu check

In [ ]:
import tensorflow as tf
from tensorflow.python.client import device_lib
tf.test.gpu_device_name()
device_lib.list_local_devices()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
### ⚠️ **Status of this notebook**

**This is a complete implementation that has NOT been run against the full dataset yet.**
Every cell below is written and ready to execute in Colab, but all outputs are cleared and
**no result numbers are filled in anywhere** - no Dice values, no Jaccard table, no loss
curves, no training logs. The cells show the code that *would* produce those numbers.
Do not read any figure in this notebook as a measured result, because there are none.

Run order: `gpu check` -> `drive mount` -> `Import library` -> `Dataset paths` ->
`Data loading` -> `EDA` (this one also sets `POS_WEIGHTS` used by the loss) ->
`Metrics and loss` -> `Build multi-label U-net` -> `Model training` -> `Evaluation`.

---

# **Task 2 - Lesion Attribute Detection**

Task 1 was one question per pixel: *is this pixel lesion or skin?*
Task 2 asks five questions per pixel at the same time: *which dermoscopic
attributes are present here?* The five attributes defined by the challenge are

| # | attribute | ground-truth file suffix |
|---|-----------|--------------------------|
| 1 | pigment network   | `_attribute_pigment_network.png` |
| 2 | negative network  | `_attribute_negative_network.png` |
| 3 | streaks           | `_attribute_streaks.png` |
| 4 | milia-like cysts  | `_attribute_milia_like_cyst.png` (singular in the filename) |
| 5 | globules (incl. dots) | `_attribute_globules.png` |

Ground truth lives in `ISIC2018_Task2_Training_GroundTruth_v3/` and every mask is named
`ISIC_<image_id>_attribute_<attribute_name>.png`, a binary PNG with exactly the same
dimensions as the lesion image. The **input images are the same 2594 images used for Task 1**
(`ISIC2018_Task1-2_Training_Input`), so I can reuse the `train_t12` / `val_t12` folders I
already uploaded for task-1 and only need to add the Task 2 ground truth. 2594 images x 5
attributes = 12970 masks.

The official metric is the **Jaccard index**, but computed differently from Task 1: images
are standardised to 256x256 and then *all prediction pixels across the entire dataset* are
pooled and compared to ground truth, rather than scoring image-by-image and averaging.
The challenge says why: many images have no positive instance of an attribute at all, and a
per-image Jaccard is undefined (or trivially 0/1) for those.
Source: <https://challenge.isic-archive.com/landing/2018/46/>

### Why this is much harder than Task 1

1. **Multi-label, not mutually exclusive.** A pigment network and globules can occupy the
   *same* pixel. So the output is 5 independent binary maps, not a 5-way classification.
2. **Extreme class imbalance.** In Task 1 the lesion covers a big chunk of the frame. Here
   the positive pixels for an attribute like streaks are a thin sliver of the image, so a
   model that predicts all-zero already gets >99% pixel accuracy. Plain BCE happily
   converges to exactly that.
3. **Many empty masks.** Most images simply do not contain most attributes; the mask file
   exists but is entirely black. The reported training frequencies are pigment network
   58.67% (1522 images), milia-like cysts 26.25% (681), globules 23.21% (602), negative
   network 7.32% (190), streaks 3.86% (100 images) - see the TATL paper,
   <https://arxiv.org/abs/2104.01641>. The EDA cell below recomputes this from my own copy
   of the data instead of trusting those numbers.
4. **Faint, low-contrast, texture-like targets.** Attribute boundaries are not real edges,
   they are pattern changes. Annotator agreement is much lower than for lesion boundaries.
5. **Small structures survive resizing badly.** Milia-like cysts and dots are a handful of
   pixels wide; a bilinear resize to 256x256 blurs them away, so masks are resized with
   nearest-neighbour interpolation here.

# Import library

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, ReduceLROnPlateau, EarlyStopping, TensorBoard
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, Precision, MeanIoU
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import CustomObjectScope,normalize
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import backend as K
import numpy as np
import pandas as pd
import random, math ,cv2
from PIL import Image
from glob import glob
from tqdm import tqdm
from skimage import io
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, precision_score, recall_score, roc_auc_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# **Dataset paths and attribute definition**

`ATTRIBUTES` is the single source of truth for the channel order of the target tensor:
channel 0 is always pigment network, channel 4 is always globules. The strings are exactly
the ones used inside the ground-truth filenames (note `milia_like_cyst`, singular).

In [ ]:
H, W = 256, 256

"""channel order of the (256, 256, 5) target - must never be reshuffled"""
ATTRIBUTES = [
    "pigment_network",
    "negative_network",
    "streaks",
    "milia_like_cyst",
    "globules",
]
N_ATTR = len(ATTRIBUTES)

"""short labels for tables/plots"""
ATTR_LABELS = {
    "pigment_network":  "Pigment network",
    "negative_network": "Negative network",
    "streaks":          "Streaks",
    "milia_like_cyst":  "Milia-like cysts",
    "globules":         "Globules",
}

"""one distinct colour per attribute, BGR because everything here goes through cv2"""
ATTR_COLORS_BGR = {
    "pigment_network":  (0, 0, 255),      # red
    "negative_network": (0, 255, 255),    # yellow
    "streaks":          (0, 255, 0),      # green
    "milia_like_cyst":  (255, 0, 255),    # magenta
    "globules":         (255, 128, 0),    # blue-cyan
}

"""2018 dataset - the images are the SAME files task-1 used (Task1-2 input)"""
train_img_dir = '/content/drive/MyDrive/ISIC2018/Dataset/t1_train_images/train_t12'
val_img_dir   = '/content/drive/MyDrive/ISIC2018/Dataset/t1_val_images/val_t12'
test_img_dir  = '/content/drive/MyDrive/ISIC2018/Dataset/t1_test_images/test_t12'

"""only the ground truth is new for task 2"""
train_gt_dir  = '/content/drive/MyDrive/ISIC2018/Dataset/t2_train_masks/ISIC2018_Task2_Training_GroundTruth_v3'
val_gt_dir    = '/content/drive/MyDrive/ISIC2018/Dataset/t2_val_masks/ISIC2018_Task2_Validation_GroundTruth'

save_dir      = '/content/drive/MyDrive/ISIC2018/Models/u_net_task2'

# **Create Dataset and metrics**

Pairing works through the image id: `ISIC_0000000.jpg` -> the five files
`ISIC_0000000_attribute_<name>.png`. A missing file is *not* an error - it is stored as an
empty string and later decoded into an all-zero channel, which is the correct semantics
(that attribute is absent from that image). This is also how the odd truncated/unreadable
PNG gets absorbed: `cv2.imread` returns `None` and we fall back to zeros.

Masks use `INTER_NEAREST` and a `> 127` threshold so that the target stays strictly binary.
Bilinear resizing (what task-1 did for its single big lesion mask) would produce grey
fringe values around these tiny structures and would quietly delete the smallest ones.

In [ ]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

# shuffle input data
def shuffling(x, y):
    x, y = shuffle(x, y, random_state=42)
    return x, y

def attribute_paths(image_id, gt_dir):
    """the 5 ground truth paths for one image id, "" when the file does not exist"""
    row = []
    for a in ATTRIBUTES:
        p = os.path.join(gt_dir, f"{image_id}_attribute_{a}.png")
        row.append(p if os.path.exists(p) else "")
    return row

def build_pairs(img_dir, gt_dir):
    """X : (N,) image paths      Y : (N, 5) mask paths in ATTRIBUTES order"""
    images = sorted(glob(os.path.join(img_dir, "*.jpg")))
    X, Y = [], []
    for p in images:
        image_id = os.path.basename(p).split(".")[0]     ## ISIC_0000000
        X.append(p)
        Y.append(attribute_paths(image_id, gt_dir))
    return np.array(X), np.array(Y)

def load_data(img_dir, gt_dir, split=0.1):
    """split on the PAIRS so an image can never drift away from its own masks"""
    X, Y = build_pairs(img_dir, gt_dir)
    test_size = int(len(X) * split)

    train_x, valid_x, train_y, valid_y = train_test_split(X, Y, test_size=test_size, random_state=42)
    train_x, test_x,  train_y, test_y  = train_test_split(train_x, train_y, test_size=test_size, random_state=42)

    return (train_x, train_y), (valid_x, valid_y), (test_x, test_y)

def read_image(path):
    path = path.decode()
    x = cv2.imread(path, cv2.IMREAD_COLOR)  ## (H, W, 3) as RGB img, no. of channel : 3
    x = cv2.resize(x, (W, H))               ## resizing
    x = x/255.0                             ## normalize, int -> float : decrease converting time
    x = x.astype(np.float32)
    return x                                ## (256, 256, 3)

def read_image_hair_removal(path):
    path = path.decode()
    src = cv2.imread(path, cv2.IMREAD_COLOR)  ## (H, W, 3) as RGB img, no. of channel : 3
    src = cv2.resize(src, (W, H))             ## resizing
    # convert to grayscale
    grayScale = cv2.cvtColor(src, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(1, (17,17))  # Kernel for the morphological filtering
    # Perform the blackHat filtering on the grayscale image to find the
    # hair countours
    blackhat = cv2.morphologyEx(grayScale, cv2.MORPH_BLACKHAT, kernel)
    # intensify the hair countours in preparation for the inpainting
    # algorithm
    ret,thresh2 = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    # inpaint the original image depending on the mask
    dst = cv2.inpaint(src, thresh2, 1, cv2.INPAINT_TELEA)
    x = dst/255.0                             ## normalize, int -> float : decrease converting time
    x = x.astype(np.float32)
    return x

def read_attribute_masks(paths):
    """5 binary masks -> one (256, 256, 5) target. missing/unreadable = all-zero channel"""
    channels = []
    for p in paths:
        if isinstance(p, bytes):
            p = p.decode()
        m = None
        if p != "":
            m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)     ## (H, W)
        if m is None:
            m = np.zeros((H, W), dtype=np.float32)      ## attribute absent from this image
        else:
            m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
            m = (m > 127).astype(np.float32)            ## keep it strictly 0/1
        channels.append(m)
    y = np.stack(channels, axis=-1)                     ## (256, 256, 5)
    return y.astype(np.float32)

def tf_parse(x, y):
    def _parse(x, y):
        # # without hair removal
        # x = read_image(x)

        # with hair removal
        x = read_image_hair_removal(x)
        y = read_attribute_masks(y)
        return x, y
    x, y = tf.numpy_function(_parse, [x, y], [tf.float32, tf.float32])
    x.set_shape([H, W, 3])
    y.set_shape([H, W, N_ATTR])
    return x, y

def tf_dataset(X, Y, batch):
    dataset = tf.data.Dataset.from_tensor_slices((X, Y))
    dataset = dataset.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch)
    dataset = dataset.prefetch(10)
    return dataset

# **Data set size check**

In [ ]:
train_images = sorted(glob(os.path.join(train_img_dir, "*.jpg")))
val_images   = sorted(glob(os.path.join(val_img_dir,   "*.jpg")))
train_masks  = sorted(glob(os.path.join(train_gt_dir,  "*.png")))
val_masks    = sorted(glob(os.path.join(val_gt_dir,    "*.png")))

print('\tTrain images:',len(train_images),'\n','\tTrain masks:',len(train_masks),'\n', '\tValidation images:',len(val_images),'\n','\tValidation masks:',len(val_masks))
print('\texpected train masks = images x attributes =',len(train_images),'x',N_ATTR,'=',len(train_images)*N_ATTR)

In [ ]:
"""sanity check on one pair: does every image really get 5 channels back?"""
X_all, Y_all = build_pairs(train_img_dir, train_gt_dir)
print('pairs:', X_all.shape, Y_all.shape)
print('images with a missing mask file:', int((Y_all == "").any(axis=1).sum()))

i = 0
xi = read_image(X_all[i].encode())
yi = read_attribute_masks(Y_all[i])
print(os.path.basename(X_all[i]), xi.shape, yi.shape, 'positive pixels per channel:', yi.reshape(-1, N_ATTR).sum(0))

# **EDA - how sparse is the ground truth?**

This is the cell that decides the loss function, so it comes before the model.
Two things are counted over the whole training set:

* **positive-pixel fraction** per attribute - what share of all pixels is positive. This is
  the number that tells you an all-zero prediction would score >99% pixel accuracy.
* **empty-mask count** per attribute - how many images have *no* positive pixel for it.

The positive fraction is then turned into the per-attribute positive weight used by the
weighted BCE term, `w = (1 - p) / p`, clipped so the rarest attribute cannot completely
dominate the gradient. Nothing is hard-coded: `POS_WEIGHTS` is whatever this cell measures.

In [ ]:
def attribute_statistics(X, Y, sample=None):
    """positive pixel fraction + empty mask count per attribute, measured at 256x256"""
    idx = np.arange(len(X))
    if sample is not None and sample < len(X):
        rng = np.random.RandomState(42)
        idx = rng.choice(idx, size=sample, replace=False)

    pos_pixels = np.zeros(N_ATTR, dtype=np.float64)
    n_empty    = np.zeros(N_ATTR, dtype=np.int64)
    n_missing  = np.zeros(N_ATTR, dtype=np.int64)
    total_pixels = 0.0

    for i in tqdm(idx, total=len(idx)):
        y = read_attribute_masks(Y[i])                 ## (256, 256, 5)
        flat = y.reshape(-1, N_ATTR)
        per_ch = flat.sum(0)
        pos_pixels += per_ch
        n_empty    += (per_ch == 0).astype(np.int64)
        n_missing  += (Y[i] == "").astype(np.int64)
        total_pixels += flat.shape[0]

    df = pd.DataFrame({
        "Attribute":       [ATTR_LABELS[a] for a in ATTRIBUTES],
        "Positive pixels": pos_pixels.astype(np.int64),
        "Positive %":      100.0 * pos_pixels / total_pixels,
        "Images with mask file": len(idx) - n_missing,
        "Images with EMPTY mask": n_empty,
        "Images WITH attribute": len(idx) - n_empty,
        "Present %":       100.0 * (len(idx) - n_empty) / len(idx),
    })
    return df, pos_pixels / total_pixels

stats_df, pos_frac = attribute_statistics(X_all, Y_all)
stats_df

In [ ]:
"""the whole reason a plain BCE will not work here"""
print('all-zero prediction would already reach a pixel accuracy of',
      f"{100.0 * (1.0 - pos_frac.mean()):.3f}%", 'averaged over the 5 channels')

"""per-attribute positive weight for the weighted BCE - measured, not guessed"""
POS_WEIGHTS = np.clip((1.0 - pos_frac) / np.maximum(pos_frac, 1e-8), 1.0, 50.0).astype(np.float32)
for a, p, w in zip(ATTRIBUTES, pos_frac, POS_WEIGHTS):
    print(f"{ATTR_LABELS[a]:<18} positive fraction {p:.6f}   ->  pos_weight {w:.2f}")

np.save(os.path.join('/content', 'task2_pos_weights.npy'), POS_WEIGHTS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))

ax[0].bar(stats_df["Attribute"], stats_df["Positive %"], color='#4c72b0')
ax[0].set_title('Positive pixels per attribute (% of all pixels)')
ax[0].set_ylabel('% of pixels')
ax[0].tick_params(axis='x', rotation=30)

ax[1].bar(stats_df["Attribute"], stats_df["Images with EMPTY mask"], color='#c44e52', label='empty mask')
ax[1].bar(stats_df["Attribute"], stats_df["Images WITH attribute"],
          bottom=stats_df["Images with EMPTY mask"], color='#55a868', label='attribute present')
ax[1].set_title('Images with / without each attribute')
ax[1].set_ylabel('images')
ax[1].tick_params(axis='x', rotation=30)
ax[1].legend()

plt.tight_layout()
plt.show()

# **Metrics and loss**

Task-1's `iou` / `dice_coef` / `dice_loss` are kept unchanged so the two notebooks stay
comparable, but on a 5-channel target they collapse everything into one global number -
which is dominated by pigment network, the one easy attribute. So the multi-label versions
below are the ones actually used:

* `dice_coef_channel(c)` / `jaccard_coef_channel(c)` - one attribute at a time.
* `dice_macro` / `jaccard_macro` - unweighted mean over the 5 attributes. Macro (not micro)
  averaging, otherwise streaks and negative network are invisible in the metric.
* `weighted_bce` - BCE with a per-channel positive weight from the EDA cell. This is what
  stops the model from collapsing to all-zero in the first few epochs.
* `combined_loss = dice_loss_multilabel + 0.5 * weighted_bce`. Soft dice gives a gradient
  that is scale-invariant with respect to how much of the image is positive (good for the
  rare attributes), weighted BCE gives a well-behaved per-pixel gradient early in training
  when dice is still nearly flat. `0.5` keeps the BCE term from taking over once the
  weights are as large as 50.

**Caveat on `dice_macro` as a training metric.** When a channel is empty in the ground truth
*and* the model correctly predicts nothing, soft dice returns `smooth/smooth = 1.0` for that
channel. Since most attributes are absent from most images, `dice_macro` therefore starts
high and is *optimistic* - it is useful for watching training move, but it is **not** a
score. The number to report is the dataset-level Jaccard computed in the evaluation section,
which pools pixels across the whole split and leaves genuinely absent attributes as `NaN`
instead of rewarding them with a 1.0.

In [ ]:
smooth = 1e-15

"""task-1 metrics, kept as-is (they treat the 5 channels as one big mask)"""
def iou(y_true, y_pred):
    def f(y_true, y_pred):
        intersection = (y_true * y_pred).sum()
        union = y_true.sum() + y_pred.sum() - intersection
        x = (intersection + 1e-15) / (union + 1e-15)
        x = x.astype(np.float32)
        return x
    return tf.numpy_function(f, [y_true, y_pred], tf.float32)

def dice_coef(y_true, y_pred):
    y_true = tf.keras.layers.Flatten()(y_true)
    y_pred = tf.keras.layers.Flatten()(y_pred)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

"""multi-label versions : one attribute channel at a time"""
def dice_coef_channel(y_true, y_pred, c):
    y_t = tf.reshape(y_true[..., c], [-1])
    y_p = tf.reshape(y_pred[..., c], [-1])
    intersection = tf.reduce_sum(y_t * y_p)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_t) + tf.reduce_sum(y_p) + smooth)

def jaccard_coef_channel(y_true, y_pred, c):
    y_t = tf.reshape(y_true[..., c], [-1])
    y_p = tf.reshape(y_pred[..., c], [-1])
    intersection = tf.reduce_sum(y_t * y_p)
    union = tf.reduce_sum(y_t) + tf.reduce_sum(y_p) - intersection
    return (intersection + smooth) / (union + smooth)

def dice_macro(y_true, y_pred):
    """unweighted mean over attributes - rare attributes count as much as pigment network"""
    return tf.add_n([dice_coef_channel(y_true, y_pred, c) for c in range(N_ATTR)]) / float(N_ATTR)

def jaccard_macro(y_true, y_pred):
    return tf.add_n([jaccard_coef_channel(y_true, y_pred, c) for c in range(N_ATTR)]) / float(N_ATTR)

"""keras needs one named callable per attribute to show it in the logs"""
def make_channel_metric(c, kind="dice"):
    fn = dice_coef_channel if kind == "dice" else jaccard_coef_channel
    def metric(y_true, y_pred):
        return fn(y_true, y_pred, c)
    metric.__name__ = f"{kind}_{ATTRIBUTES[c]}"
    return metric

per_attribute_metrics = [make_channel_metric(c, "dice") for c in range(N_ATTR)]

In [ ]:
"""POS_WEIGHTS normally comes from the EDA cell; fall back to plain BCE if it was skipped"""
if "POS_WEIGHTS" not in globals():
    print('POS_WEIGHTS not found - falling back to unweighted BCE. Run the EDA cell for the real weights.')
    POS_WEIGHTS = np.ones(N_ATTR, dtype=np.float32)

def weighted_bce(y_true, y_pred):
    """binary cross entropy with a per-attribute weight on the positive class"""
    w = tf.constant(POS_WEIGHTS, dtype=tf.float32)                 ## (5,) broadcasts over H, W
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)            ## keep log() finite
    loss = -(w * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    return tf.reduce_mean(loss)

def dice_loss_multilabel(y_true, y_pred):
    return 1.0 - dice_macro(y_true, y_pred)

def combined_loss(y_true, y_pred):
    """soft dice for the sparse structures + weighted BCE for a usable early gradient"""
    return dice_loss_multilabel(y_true, y_pred) + 0.5 * weighted_bce(y_true, y_pred)

# **Build multi-label U-net**

Same skeleton as task-1 - `conv_block` / `encoder_block` / `decoder_block`, 64-128-256-512
encoder, 1024 bridge - with exactly one change that matters:

```python
outputs = Conv2D(N_ATTR, 1, padding="same", activation="sigmoid")(d4)   # 5 channels
```

**Sigmoid, not softmax.** Softmax would force the 5 attribute scores at a pixel to sum to 1,
i.e. it would assume the attributes are mutually exclusive. They are not - the ISIC ground
truth routinely marks the same pixel as both pigment network and globules, and a pixel can
equally belong to *none* of the five. Sigmoid gives 5 independent Bernoulli outputs, so
co-occurrence and "all five absent" are both representable. A `Dropout(0.2)` sits on the
bridge because the rare attributes are easy to memorise from 100-190 training images.

In [ ]:
"""Convolutional block"""
def conv_block(inputs, num_filters):
    x = Conv2D(num_filters, 3, padding="same")(inputs)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    # added BatchNormalization and Activation 'relu' between 2 Conv2D layers
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

"""Encoder"""
def encoder_block(inputs, num_filters):
    x = conv_block(inputs, num_filters)  ## Convolution process will increase the depth of the image
    p = MaxPool2D((2, 2))(x)             ## Maxpooling halves down size of image
    return x, p

"""Decoder"""
def decoder_block(inputs, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(inputs)
    x = Concatenate()([x, skip_features])
    x = conv_block(x, num_filters)
    return x

"""Multi-label U-net definition"""
def build_unet_multilabel(input_shape, n_attr=N_ATTR):
    inputs = Input(input_shape)

    """ Encoder: Down sampling """
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    """ Bridge : The bottom Layer """
    b1 = conv_block(p4, 1024)
    b1 = Dropout(0.2)(b1)              ## streaks only has ~100 training images, it will memorise

    """ Decoder : Up sampling """
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    """ Outputs : 5 co-occurring attributes -> 5 independent sigmoids, NOT softmax """
    outputs = Conv2D(n_attr, 1, padding="same", activation="sigmoid")(d4)

    """ Model """
    model = Model(inputs, outputs, name="U-Net_task2_multilabel")
    return model

"""ResNet50-Unet - same encoder swap as task-1, 5-channel head"""
def build_resnet50_unet_multilabel(input_shape, n_attr=N_ATTR):
    inputs = Input(input_shape)

    resnet50 = ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)

    """ Encoder """
    s1 = resnet50.get_layer(index=0).output             ## (256 x 256)
    s2 = resnet50.get_layer("conv1_relu").output        ## (128 x 128)
    s3 = resnet50.get_layer("conv2_block3_out").output  ## (64 x 64)
    s4 = resnet50.get_layer("conv3_block4_out").output  ## (32 x 32)

    """ Bridge """
    b1 = resnet50.get_layer("conv4_block6_out").output  ## (16 x 16)

    """ Decoder """
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    """ Output """
    outputs = Conv2D(n_attr, 1, padding="same", activation="sigmoid")(d4)

    model = Model(inputs, outputs, name="ResNet50_U-Net_task2_multilabel")
    return model

In [ ]:
model = build_unet_multilabel((H, W, 3))
model.summary()
print('output shape:', model.output_shape, '  <- (None, 256, 256, 5)')

#**Model training**

Callbacks are the same set task-1 used, only the monitored quantity changes: `val_loss` is
still the checkpoint criterion, and `val_dice_macro` is logged so I can see whether the
rare attributes are moving at all rather than watching an average that pigment network
carries. `lr = 1e-4` with `ReduceLROnPlateau` down to `1e-7`.

In [ ]:
create_dir(save_dir)
%cd /content/drive/MyDrive/ISIC2018/Models/u_net_task2
%load_ext tensorboard
%tensorboard --logdir logs/

In [ ]:
if __name__ == "__main__":
    """ Seeding : make result reproducible"""
    np.random.seed(42)
    tf.random.set_seed(42)

    """ Folder for saving data """
    create_dir(save_dir)

    """ Hyperparameters """
    batch_size = 8
    lr = 1e-4
    num_epoch = 40
    H, W = 256, 256

    """output dir"""
    model_path = f"{save_dir}/u_net_task2.h5"
    csv_path   = f"{save_dir}/u_net_task2.csv"
    log_dir    = f"{save_dir}/logs"

    """Load dataset - X is (N,) image paths, Y is (N, 5) attribute mask paths"""
    (train_x, train_y), (valid_x, valid_y), (test_x, test_y) = load_data(train_img_dir, train_gt_dir)
    train_x, train_y = shuffling(train_x, train_y)

    train_dataset = tf_dataset(train_x, train_y, batch_size)
    valid_dataset = tf_dataset(valid_x, valid_y, batch_size)
    test_dataset  = tf_dataset(test_x,  test_y,  batch_size)

    """step size"""
    train_steps = len(train_x)//batch_size
    valid_steps = len(valid_x)//batch_size

    """do not forget about the reamainder"""
    if len(train_x) % batch_size != 0:
        train_steps += 1

    if len(valid_x) % batch_size != 0:
        valid_steps += 1

    """Summary"""
    print(f"Train: {len(train_x)} - {len(train_y)}")
    print(f"Valid: {len(valid_x)} - {len(valid_y)}")
    print(f"Test: {len(test_x)} - {len(test_y)}")

    """ Build Model """
    model = build_unet_multilabel((H, W, 3))

    # """reload model instead of rebuilding"""
    # with CustomObjectScope({'combined_loss': combined_loss, 'dice_macro': dice_macro,
    #                         'jaccard_macro': jaccard_macro, 'weighted_bce': weighted_bce,
    #                         'dice_loss_multilabel': dice_loss_multilabel}):
    #     model = tf.keras.models.load_model(model_path)

    metrics = [dice_macro, jaccard_macro, Recall(), Precision()] + per_attribute_metrics
    model.compile(loss=combined_loss, optimizer=Adam(lr), metrics=metrics)

    callbacks = [
                    ModelCheckpoint(filepath=model_path, monitor='val_loss', verbose=1, save_best_only=True),
                    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=4, min_lr=1e-7, verbose=1),
                    CSVLogger(csv_path),
                    TensorBoard(log_dir=log_dir),
                    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=False)
                ]

    """Model.fit without augmentation"""
    history = model.fit(
                            train_dataset,
                            epochs = num_epoch,
                            validation_data = valid_dataset,
                            steps_per_epoch = train_steps,
                            validation_steps = valid_steps,
                            callbacks = callbacks
                        )

In [ ]:
"""training curves - reads back the CSVLogger file, nothing is plotted until it exists"""
csv_path = f"{save_dir}/u_net_task2.csv"
df = pd.read_csv(csv_path)

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(df['epoch'], df['loss'], label='train')
ax[0].plot(df['epoch'], df['val_loss'], label='valid')
ax[0].set_title('combined loss'); ax[0].set_xlabel('epoch'); ax[0].legend()

for a in ATTRIBUTES:
    col = f"val_dice_{a}"
    if col in df.columns:
        ax[1].plot(df['epoch'], df[col], label=ATTR_LABELS[a])
ax[1].set_title('validation Dice per attribute'); ax[1].set_xlabel('epoch'); ax[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# **Model Validation - per attribute Jaccard**

This follows the official Task 2 definition rather than task-1's per-image averaging:
predictions and ground truth are accumulated at 256x256 into one intersection and one union
counter **per attribute over the whole dataset**, and the Jaccard is computed at the end
from those totals. Quoting the task page: *"all prediction pixels across the entire dataset
(not image-by-image, as some images may have no positive instances of a dermascopic
attribute) are compared to ground truth pixels."*

Two consequences worth knowing before reading the table:

* An attribute with no positive ground-truth pixel anywhere in the split has union 0; that
  row is reported as `NaN`, not 0, and is dropped from the macro mean.
* The macro mean over the 5 attributes is the headline number, and it is the one directly
  comparable to the published challenge results in the last section.

In [ ]:
H, W = 256, 256
output_dir = f"{save_dir}/Validation_result"
model_name = 'u_net_task2.h5'

def load_validation_dataset(img_dir, gt_dir):
    X, Y = build_pairs(img_dir, gt_dir)
    return (X, Y)

def read_eval_image(path):
    x = cv2.imread(path, cv2.IMREAD_COLOR)  ## (H, W, 3)
    x = cv2.resize(x, (W, H))
    ori_x = x
    x = x/255.0
    x = x.astype(np.float32)
    x = np.expand_dims(x, axis=0)
    return ori_x, x                         ## (1, 256, 256, 3)

def load_task2_model(path):
    """compile=False, the custom loss/metrics are not needed just to predict"""
    return tf.keras.models.load_model(path, compile=False)

def evaluate_attributes(img_dir, gt_dir, model_path, thresholds=None, save_csv=None):
    """official-style metric : pool every pixel in the dataset, then divide"""
    np.random.seed(42)
    tf.random.set_seed(42)

    if thresholds is None:
        thresholds = np.full(N_ATTR, 0.5, dtype=np.float32)

    model = load_task2_model(model_path)
    (test_x, test_y) = load_validation_dataset(img_dir, gt_dir)

    inter = np.zeros(N_ATTR, dtype=np.float64)
    union = np.zeros(N_ATTR, dtype=np.float64)
    tp    = np.zeros(N_ATTR, dtype=np.float64)
    fp    = np.zeros(N_ATTR, dtype=np.float64)
    fn    = np.zeros(N_ATTR, dtype=np.float64)

    for xp, yp in tqdm(zip(test_x, test_y), total=len(test_x)):
        ori_x, x = read_eval_image(xp)
        y = read_attribute_masks(yp)                             ## (256, 256, 5) in {0,1}

        prob = model.predict(x, verbose=0)[0]                    ## (256, 256, 5)
        pred = (prob >= thresholds.reshape(1, 1, N_ATTR)).astype(np.float32)

        y_f = y.reshape(-1, N_ATTR)
        p_f = pred.reshape(-1, N_ATTR)

        i_ch = (y_f * p_f).sum(0)
        inter += i_ch
        union += y_f.sum(0) + p_f.sum(0) - i_ch
        tp    += i_ch
        fp    += ((1.0 - y_f) * p_f).sum(0)
        fn    += (y_f * (1.0 - p_f)).sum(0)

    """union == 0 means the attribute is absent from ground truth AND prediction -> undefined"""
    jaccard = np.where(union > 0, inter / np.maximum(union, 1e-12), np.nan)
    dice    = np.where((2*tp + fp + fn) > 0, 2*tp / np.maximum(2*tp + fp + fn, 1e-12), np.nan)
    prec    = np.where((tp + fp) > 0, tp / np.maximum(tp + fp, 1e-12), np.nan)
    rec     = np.where((tp + fn) > 0, tp / np.maximum(tp + fn, 1e-12), np.nan)

    df = pd.DataFrame({
        "Attribute": [ATTR_LABELS[a] for a in ATTRIBUTES],
        "Threshold": thresholds,
        "Jaccard":   jaccard,
        "Dice":      dice,
        "Precision": prec,
        "Recall":    rec,
    })
    df.loc[len(df)] = ["MACRO AVERAGE", np.nan, np.nanmean(jaccard), np.nanmean(dice),
                       np.nanmean(prec), np.nanmean(rec)]

    if save_csv is not None:
        create_dir(os.path.dirname(save_csv))
        df.to_csv(save_csv, index=False)
    return df

In [ ]:
result_df = evaluate_attributes(
    img_dir    = val_img_dir,
    gt_dir     = val_gt_dir,
    model_path = f"{save_dir}/{model_name}",
    save_csv   = f"{output_dir}/task2_attribute_scores.csv",
)
result_df

# **Per-attribute threshold search**

0.5 is the wrong operating point for a heavily weighted sigmoid: the positive weighting
pushes the probabilities up, and each attribute has a different prevalence, so each one
wants its own threshold. This sweeps a grid per attribute and keeps whichever value
maximises that attribute's dataset-level Jaccard. It must be fitted on the validation split
only, then reused unchanged on the test split.

In [ ]:
def tune_thresholds(img_dir, gt_dir, model_path, grid=np.arange(0.05, 0.96, 0.05)):
    """cache the probabilities once, then score every threshold offline"""
    model = load_task2_model(model_path)
    (test_x, test_y) = load_validation_dataset(img_dir, gt_dir)

    inter = np.zeros((len(grid), N_ATTR), dtype=np.float64)
    union = np.zeros((len(grid), N_ATTR), dtype=np.float64)

    for xp, yp in tqdm(zip(test_x, test_y), total=len(test_x)):
        ori_x, x = read_eval_image(xp)
        y_f = read_attribute_masks(yp).reshape(-1, N_ATTR)
        prob = model.predict(x, verbose=0)[0].reshape(-1, N_ATTR)

        for gi, t in enumerate(grid):
            p_f = (prob >= t).astype(np.float32)
            i_ch = (y_f * p_f).sum(0)
            inter[gi] += i_ch
            union[gi] += y_f.sum(0) + p_f.sum(0) - i_ch

    jac = np.where(union > 0, inter / np.maximum(union, 1e-12), np.nan)   ## (len(grid), 5)
    best_idx = np.nanargmax(np.nan_to_num(jac, nan=-1.0), axis=0)
    best_thresholds = grid[best_idx].astype(np.float32)

    curve = pd.DataFrame(jac, index=np.round(grid, 2),
                         columns=[ATTR_LABELS[a] for a in ATTRIBUTES])
    curve.index.name = 'threshold'
    return best_thresholds, curve

best_thresholds, curve_df = tune_thresholds(val_img_dir, val_gt_dir, f"{save_dir}/{model_name}")
print('best threshold per attribute:', dict(zip(ATTRIBUTES, best_thresholds)))

curve_df.plot(figsize=(9, 4), marker='o')
plt.title('dataset-level Jaccard vs decision threshold')
plt.xlabel('threshold'); plt.ylabel('Jaccard'); plt.grid(alpha=.3)
plt.show()

In [ ]:
"""re-score with the tuned thresholds"""
tuned_df = evaluate_attributes(
    img_dir    = val_img_dir,
    gt_dir     = val_gt_dir,
    model_path = f"{save_dir}/{model_name}",
    thresholds = best_thresholds,
    save_csv   = f"{output_dir}/task2_attribute_scores_tuned.csv",
)
tuned_df

# **Qualitative visualization**

Five overlapping binary masks cannot be shown as a grey image the way task-1 did it, so
each attribute gets its own colour and is composited onto the lesion image: ground truth is
drawn as a translucent fill, prediction as a solid contour, so agreement shows up as a
coloured region wrapped in its own colour outline.

* pigment network - red
* negative network - yellow
* streaks - green
* milia-like cysts - magenta
* globules - blue

The right-hand column is the per-channel strip, which is usually the more honest view: it
shows which attributes the model simply left empty.

In [ ]:
def overlay_masks(ori_bgr, masks, alpha=0.45, mode='fill'):
    """masks : (256, 256, 5) in {0,1}. mode 'fill' = translucent, 'contour' = outline"""
    out = ori_bgr.copy().astype(np.uint8)
    for c, a in enumerate(ATTRIBUTES):
        m = (masks[..., c] > 0.5).astype(np.uint8)
        if m.sum() == 0:
            continue
        color = ATTR_COLORS_BGR[a]
        if mode == 'fill':
            layer = out.copy()
            layer[m == 1] = color
            out = cv2.addWeighted(layer, alpha, out, 1 - alpha, 0)
        else:
            contours, _ = cv2.findContours(m, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(out, contours, -1, color, 1)
    return out

def bgr2rgb(x):
    return cv2.cvtColor(x.astype(np.uint8), cv2.COLOR_BGR2RGB)

def visualize_predictions(img_dir, gt_dir, model_path, thresholds=None, rows=4, seed=None):
    model = load_task2_model(model_path)
    X, Y = build_pairs(img_dir, gt_dir)
    if thresholds is None:
        thresholds = np.full(N_ATTR, 0.5, dtype=np.float32)

    n = random.randint(0, len(X) - rows - 1) if seed is None else seed

    handles = [mpatches.Patch(color=np.array(ATTR_COLORS_BGR[a][::-1])/255.0,
                              label=ATTR_LABELS[a]) for a in ATTRIBUTES]

    for i in range(rows):
        ori_x, x = read_eval_image(X[n])
        y    = read_attribute_masks(Y[n])
        prob = model.predict(x, verbose=0)[0]
        pred = (prob >= thresholds.reshape(1, 1, N_ATTR)).astype(np.float32)

        gt_ov   = overlay_masks(ori_x, y,    mode='fill')
        pr_ov   = overlay_masks(ori_x, pred, mode='fill')
        both_ov = overlay_masks(overlay_masks(ori_x, y, mode='fill'), pred, mode='contour')

        fig, ax = plt.subplots(1, 4 + N_ATTR, figsize=(26, 3.4))
        ax[0].imshow(bgr2rgb(ori_x));  ax[0].set_title(os.path.basename(X[n]).split('.')[0], fontsize=9)
        ax[1].imshow(bgr2rgb(gt_ov));  ax[1].set_title('Ground truth', fontsize=9)
        ax[2].imshow(bgr2rgb(pr_ov));  ax[2].set_title('Prediction', fontsize=9)
        ax[3].imshow(bgr2rgb(both_ov)); ax[3].set_title('GT fill + pred contour', fontsize=9)

        for c, a in enumerate(ATTRIBUTES):
            """green = correct, red = false positive, blue = missed"""
            cmp = np.zeros((H, W, 3), dtype=np.uint8)
            yt, yp = y[..., c] > 0.5, pred[..., c] > 0.5
            cmp[np.logical_and(yt, yp)]  = (0, 255, 0)
            cmp[np.logical_and(~yt, yp)] = (255, 0, 0)
            cmp[np.logical_and(yt, ~yp)] = (0, 0, 255)
            ax[4 + c].imshow(cmp)
            ax[4 + c].set_title(f"{ATTR_LABELS[a]}\nGT px {int(yt.sum())} / pred px {int(yp.sum())}", fontsize=8)

        for a_ in ax:
            a_.axis('off')
        ax[3].legend(handles=handles, loc='upper right', fontsize=6, framealpha=.8)
        plt.tight_layout()
        plt.show()
        n += 1

visualize_predictions(val_img_dir, val_gt_dir, f"{save_dir}/{model_name}", rows=4)

# **Test model**

In [ ]:
predict_dir = f"{save_dir}/Prediction_result"

def save_attribute_predictions(img_dir, model_path, out_dir, thresholds=None):
    """write one PNG per attribute per image, in the exact submission naming scheme"""
    create_dir(out_dir)
    model = load_task2_model(model_path)
    if thresholds is None:
        thresholds = np.full(N_ATTR, 0.5, dtype=np.float32)

    images = sorted(glob(os.path.join(img_dir, "*.jpg")))
    for p in tqdm(images, total=len(images)):
        image_id = os.path.basename(p).split('.')[0]
        src = cv2.imread(p, cv2.IMREAD_COLOR)
        h0, w0 = src.shape[:2]                              ## masks must match the ORIGINAL size

        x = cv2.resize(src, (W, H)).astype(np.float32) / 255.0
        prob = model.predict(np.expand_dims(x, axis=0), verbose=0)[0]

        for c, a in enumerate(ATTRIBUTES):
            m = (prob[..., c] >= thresholds[c]).astype(np.uint8) * 255
            m = cv2.resize(m, (w0, h0), interpolation=cv2.INTER_NEAREST)
            cv2.imwrite(os.path.join(out_dir, f"{image_id}_attribute_{a}.png"), m)

save_attribute_predictions(test_img_dir, f"{save_dir}/{model_name}", predict_dir, thresholds=best_thresholds)

# **Expected difficulty**

Before reading any number this notebook eventually produces, calibrate against what the
challenge itself achieved, because Task 2 scores are *low* in absolute terms and a modest
result here is normal rather than a bug.

The ISIC 2018 Task 2 winner (Koohbanani et al., an ensemble of four ImageNet-pretrained
encoders - ResNet152, DenseNet169, Xception, InceptionResNetV2 - inside U-Nets) scored:

| attribute | winning Jaccard |
|-----------|-----------------|
| Pigment network  | 0.563 |
| Globules         | 0.341 |
| Negative network | 0.228 |
| Milia-like cysts | 0.171 |
| Streaks          | 0.156 |
| **average**      | **0.292** |

Source: results table in *TATL: Task Agnostic Transfer Learning for Skin Attributes
Detection*, Le et al., Medical Image Analysis 2022 - <https://arxiv.org/abs/2104.01641>
(the same paper reports the training-set frequencies quoted in the intro).
For scale, the Task 1 leaderboard at <https://challenge.isic-archive.com/leaderboards/2018/>
tops out around **0.80** Jaccard for lesion boundary segmentation - so the *best in the
world* at attribute detection is roughly a third of what a plain U-Net gets on Task 1.

What that implies for this notebook:

* A **macro Jaccard in the 0.1-0.2 range for a single unpretrained U-Net is a reasonable
  outcome, not a broken model.** Beating 0.292 with one 256x256 U-Net trained on Colab
  would be surprising.
* The ranking across attributes is expected to be lopsided in the same direction: pigment
  network by far the best (it is large, textured and present in ~59% of images), then
  globules, with **streaks and negative network far worse** - they have ~100 and ~190
  positive training images respectively, and even the winner stayed near 0.15-0.23 on them.
* If streaks or negative network come out at exactly 0.0, check first whether the model
  simply predicts all-zero for that channel (the per-channel strip in the visualization
  cell shows this immediately) before assuming a code bug. That is the failure mode the
  weighted BCE and the per-attribute threshold search exist to fight.
* Annotator noise is part of the ceiling: these attributes are pattern judgements, not
  object boundaries, so the ground truth itself is soft.

### Things worth trying next
1. Pretrained encoder (`build_resnet50_unet_multilabel` is already here) - this is the
   single biggest gap versus the winning entry.
2. Train at higher resolution or on crops; milia-like cysts and dots lose most of their
   pixels at 256x256.
3. Per-attribute heads or one model per attribute, so streaks stops competing for capacity
   with pigment network.
4. Focal / Tversky loss instead of the dice + weighted BCE combination used here.
5. Test-time augmentation (the 4-flip scheme from task-1) applied per channel.